In [1]:
import sys, os
os.chdir('..')
sys.path.insert(0, '.')

import nltk
nltk.download('punkt_tab', quiet=True)

print("Setup complete!")

Setup complete!


In [2]:
# This is the correct way — one import, everything available
from pipeline import (
    analyze_review,
    extract_aspects,
    clean_text,
    split_into_sentences,
    get_sentiment_for_sentence,
    ASPECT_VOCAB,
    WORD_TO_ASPECT
)

print("Pipeline imported successfully")
print(f"Aspects tracked: {list(ASPECT_VOCAB.keys())}")

d:\Project 2\.venv1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8848.92it/s]


Pipeline imported successfully
Aspects tracked: ['battery', 'screen', 'camera', 'price', 'delivery']


In [3]:
# Demo

review = (
    "The battery life is terrible — dies in 2 hours. "
    "But the screen looks absolutely amazing. "
    "The price is reasonable for what you get."
)

result = analyze_review(review)

print("The Core Insight Demo")
print(f"\nReview:\n  '{review}'\n")
print(f"Overall sentiment: {result['overall']['sentiment'].upper()} "
      f"({result['overall']['confidence']:.0%} confidence)")
print("\nAspect breakdown:")
print("-" * 40)

for aspect, data in result['aspects'].items():
    emoji = "✅" if data['sentiment'] == 'positive' else \
            "❌" if data['sentiment'] == 'negative' else "➖"
    print(f"{emoji}  {aspect.upper()}: {data['sentiment']} "
          f"({data['confidence']:.0%})")
    print(f"     Evidence: '{data['sentence']}'")

print("\n→ A neutral overall score was hiding a strong negative")
print("  (battery) and a strong positive (screen).")

The Core Insight Demo

Review:
  'The battery life is terrible — dies in 2 hours. But the screen looks absolutely amazing. The price is reasonable for what you get.'

Overall sentiment: POSITIVE (35% confidence)

Aspect breakdown:
----------------------------------------
❌  BATTERY: negative (77%)
     Evidence: 'The battery life is terrible — dies in 2 hours.'
✅  SCREEN: positive (41%)
     Evidence: 'But the screen looks absolutely amazing.'
✅  PRICE: positive (57%)
     Evidence: 'The price is reasonable for what you get.'

→ A neutral overall score was hiding a strong negative
  (battery) and a strong positive (screen).


In [4]:
test_reviews = [
    {
        "text": "Absolutely love the camera quality! Photos are stunning. "
                "However the delivery took 2 weeks and packaging was terrible.",
        "expected": "camera=positive, delivery=negative"
    },
    {
        "text": "Screen brightness is perfect outdoors. "
                "Battery charges quickly too. Great value for the price.",
        "expected": "screen=positive, battery=positive, price=positive"
    },
    {
        "text": "The display is dim and hard to read. "
                "Way too expensive for what you get. "
                "Camera is decent though.",
        "expected": "screen=negative, price=negative, camera=neutral/positive"
    }
]

for i, case in enumerate(test_reviews, 1):
    print()
    print(f"Review {i}: {case['text'][:60]}...")
    print(f"Expected:  {case['expected']}")
    print(f"Got:")

    result = analyze_review(case['text'])
    print(f"  Overall: {result['overall']['sentiment']} "
          f"({result['overall']['confidence']:.0%})")

    for aspect, data in result['aspects'].items():
        print(f"  {aspect}: {data['sentiment']} ({data['confidence']:.0%})")


Review 1: Absolutely love the camera quality! Photos are stunning. How...
Expected:  camera=positive, delivery=negative
Got:
  Overall: positive (98%)
  camera: positive (99%)
  delivery: negative (65%)

Review 2: Screen brightness is perfect outdoors. Battery charges quick...
Expected:  screen=positive, battery=positive, price=positive
Got:
  Overall: positive (52%)
  screen: neutral (49%)
  battery: positive (44%)
  price: positive (99%)

Review 3: The display is dim and hard to read. Way too expensive for w...
Expected:  screen=negative, price=negative, camera=neutral/positive
Got:
  Overall: neutral (48%)
  screen: negative (46%)
  camera: neutral (64%)


In [5]:
# Show the difference between the two models side by side

review = (
    "The battery life is absolutely dreadful. "
    "Screen display is gorgeous though. "
    "Fast delivery and great packaging."
)

print("Comparing Baseline vs Upgrade on same review")
print(f"Review: '{review}'\n")

for model_type, label in [('upgrade', 'Sentence-Transformers + XGBoost'),
                           ('baseline', 'TF-IDF + Logistic Regression')]:
    use_upgrade = (model_type == 'upgrade')
    result = analyze_review(review, use_upgrade=use_upgrade)

    print(f"Model: {label}")
    print(f"  Overall: {result['overall']['sentiment']} "
          f"({result['overall']['confidence']:.0%})")
    for aspect, data in result['aspects'].items():
        print(f"  {aspect}: {data['sentiment']} ({data['confidence']:.0%})")
    print()

Comparing Baseline vs Upgrade on same review
Review: 'The battery life is absolutely dreadful. Screen display is gorgeous though. Fast delivery and great packaging.'

Model: Sentence-Transformers + XGBoost
  Overall: positive (47%)
  battery: neutral (66%)
  screen: neutral (40%)
  delivery: positive (99%)

Model: TF-IDF + Logistic Regression
  Overall: positive (81%)
  battery: negative (40%)
  screen: positive (46%)
  delivery: positive (90%)



In [6]:
from datasets import load_dataset
import pandas as pd

print("Loading sample for bulk testing")

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    trust_remote_code=True,
    streaming=True
)

bulk_records = []
for i, row in enumerate(dataset['full']):
    if i >= 100:
        break
    bulk_records.append({'text': row['text'], 'rating': row['rating']})

bulk_df = pd.DataFrame(bulk_records).dropna()

# Run pipeline on each review
results_list = []
for _, row in bulk_df.iterrows():
    res = analyze_review(str(row['text']))
    results_list.append({
        'rating':           row['rating'],
        'overall_sentiment':res['overall']['sentiment'],
        'aspects_found':    list(res['aspects'].keys()),
        'aspect_count':     len(res['aspects'])
    })

results_df = pd.DataFrame(results_list)

print("\nSentiment distribution across 100 reviews:")
print(results_df['overall_sentiment'].value_counts())
print(f"\nAverage aspects detected per review: "
      f"{results_df['aspect_count'].mean():.2f}")
print(f"\nReviews with aspects detected: "
      f"{(results_df['aspect_count'] > 0).sum()} / {len(results_df)}")

Loading sample for bulk testing

Sentiment distribution across 100 reviews:
overall_sentiment
positive    76
negative    17
neutral      7
Name: count, dtype: int64

Average aspects detected per review: 0.26

Reviews with aspects detected: 23 / 100


In [8]:
import os
project_root = os.path.dirname(os.path.abspath("__file__"))

required_files = [
    'models/baseline_model.pkl',
    'models/xgb_model.pkl',
    'models/label_encoder.pkl',
    'pipeline.py'
]

print(f"Checking inside: {project_root}\n")

all_good = True
for f in required_files:
    full_path = os.path.join(project_root, f)
    exists = os.path.exists(full_path)
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {full_path}")
    if not exists:
        all_good = False

print()
if all_good:
    print("All files present. Ready to build the Streamlit app!")
else:
    print("Some files are missing. Check paths above.")

Checking inside: d:\Project 2

  ✅  d:\Project 2\models/baseline_model.pkl
  ✅  d:\Project 2\models/xgb_model.pkl
  ✅  d:\Project 2\models/label_encoder.pkl
  ✅  d:\Project 2\pipeline.py

All files present. Ready to build the Streamlit app!
